# Model Training History
Loss, accuracy, and F1 score across training epochs.

In [ ]:
import json
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import warnings
warnings.filterwarnings('ignore')

# ── Load data ──────────────────────────────────────────────────────────────────
with open('training_history.json') as f:
    history = json.load(f)

epochs     = [d['epoch']     for d in history]
train_loss = [d['train_loss'] for d in history]
val_loss   = [d['val_loss']   for d in history]
train_acc  = [d['train_acc']  for d in history]
val_acc    = [d['val_acc']    for d in history]
val_f1     = [d['val_f1']     for d in history]

print(f'Loaded {len(history)} epochs')
print(f'Best val loss : {min(val_loss):.4f}  (epoch {epochs[val_loss.index(min(val_loss))]})')
print(f'Best val acc  : {max(val_acc):.4f}  (epoch {epochs[val_acc.index(max(val_acc))]})')
print(f'Best val F1   : {max(val_f1):.4f}  (epoch {epochs[val_f1.index(max(val_f1))]})')

In [ ]:
# ── Palette & shared style ─────────────────────────────────────────────────────
# Blues
BLUE_LIGHT  = '#D4E1F5'
BLUE        = '#7EA6E0'
# Purples
PURPLE_LIGHT = '#E1D5E7'
PURPLE       = '#A680B8'
# Reds
RED_LIGHT   = '#F8CECC'
RED         = '#EA6B66'
# Greens
GREEN_LIGHT = '#D5E8D4'
GREEN       = '#97D077'
# Neutrals
BLACK       = '#1A1A1A'
GRAY_DARK   = '#555555'
GRAY_MID    = '#999999'
GRAY_LIGHT  = '#DDDDDD'
GRAY_FAINT  = '#F5F5F5'
WHITE       = '#FFFFFF'

plt.rcParams.update({
    'figure.facecolor':    WHITE,
    'axes.facecolor':      WHITE,
    'axes.edgecolor':      GRAY_LIGHT,
    'axes.labelcolor':     GRAY_DARK,
    'axes.labelsize':      10,
    'axes.titlesize':      12,
    'axes.titleweight':    'bold',
    'axes.titlepad':       12,
    'axes.spines.top':     False,
    'axes.spines.right':   False,
    'xtick.color':         GRAY_MID,
    'ytick.color':         GRAY_MID,
    'xtick.labelsize':     9,
    'ytick.labelsize':     9,
    'text.color':          BLACK,
    'grid.color':          GRAY_FAINT,
    'grid.linewidth':      1.0,
    'legend.frameon':      True,
    'legend.framealpha':   1.0,
    'legend.edgecolor':    GRAY_LIGHT,
    'legend.fontsize':     9,
    'font.family':         'monospace',
    'figure.dpi':          120,
})

def style_ax(ax, title, xlabel='Epoch', ylabel=''):
    ax.set_title(title, color=BLACK, loc='left')
    ax.set_xlabel(xlabel, color=GRAY_MID)
    ax.set_ylabel(ylabel, color=GRAY_MID)
    ax.xaxis.set_major_locator(ticker.MaxNLocator(integer=True))
    ax.grid(True, linestyle='-', alpha=1.0)
    ax.spines['left'].set_color(GRAY_LIGHT)
    ax.spines['bottom'].set_color(GRAY_LIGHT)

def mark_best(ax, x_vals, y_vals, mode='min', label_fmt='.4f'):
    idx = y_vals.index(min(y_vals) if mode == 'min' else max(y_vals))
    bx, by = x_vals[idx], y_vals[idx]
    ax.axvline(bx, color=GRAY_MID, lw=1.0, ls=':', alpha=0.8, zorder=2)
    ax.scatter([bx], [by], color=BLACK, s=70, zorder=6,
               edgecolors=WHITE, linewidths=1.5)
    ax.annotate(f'best  {by:{label_fmt}}',
                xy=(bx, by), xytext=(10, 6), textcoords='offset points',
                fontsize=8, color=GRAY_DARK, fontweight='bold',
                arrowprops=dict(arrowstyle='->', color=GRAY_MID, lw=0.9))

print('Style configured.')

## Training & Validation Loss

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4.5))

ax.fill_between(epochs, train_loss, color=BLUE_LIGHT, alpha=1.0)
ax.fill_between(epochs, val_loss,   color=RED_LIGHT,  alpha=1.0)
ax.plot(epochs, train_loss, 'o-', color=BLUE,  lw=2, ms=5, label='Train')
ax.plot(epochs, val_loss,   's-', color=RED,   lw=2, ms=5, label='Validation')
mark_best(ax, epochs, val_loss, mode='min')
ax.legend()
style_ax(ax, 'Loss', ylabel='Loss')

plt.tight_layout()
plt.savefig('plot_loss.png', dpi=150, bbox_inches='tight')
plt.show()

## Training & Validation Accuracy

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4.5))

ax.fill_between(epochs, train_acc, color=BLUE_LIGHT, alpha=1.0)
ax.fill_between(epochs, val_acc,   color=RED_LIGHT,  alpha=1.0)
ax.plot(epochs, train_acc, 'o-', color=BLUE, lw=2, ms=5, label='Train')
ax.plot(epochs, val_acc,   's-', color=RED,  lw=2, ms=5, label='Validation')
mark_best(ax, epochs, val_acc, mode='max', label_fmt='.2%')
ax.yaxis.set_major_formatter(ticker.FuncFormatter(lambda y, _: f'{y:.0%}'))
ax.set_ylim(max(0, min(train_acc + val_acc) - 0.04), 1.01)
ax.legend()
style_ax(ax, 'Accuracy', ylabel='Accuracy')

plt.tight_layout()
plt.savefig('plot_accuracy.png', dpi=150, bbox_inches='tight')
plt.show()

## Validation F1 Score

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4.5))

ax.fill_between(epochs, val_f1, color=PURPLE_LIGHT, alpha=1.0)
ax.plot(epochs, val_f1, 'D-', color=PURPLE, lw=2, ms=5, label='Validation F1')
mark_best(ax, epochs, val_f1, mode='max')
ax.legend()
style_ax(ax, 'Validation F1 Score', ylabel='F1 Score')

plt.tight_layout()
plt.savefig('plot_f1.png', dpi=150, bbox_inches='tight')
plt.show()

## Train–Validation Loss Gap

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4.5))

gap = [tl - vl for tl, vl in zip(train_loss, val_loss)]
bar_fill   = [BLUE_LIGHT if g >= 0 else RED_LIGHT for g in gap]
bar_border = [BLUE       if g >= 0 else RED       for g in gap]
bars = ax.bar(epochs, gap, color=bar_fill, edgecolor=bar_border,
              linewidth=1.2, width=0.55)
ax.axhline(0, color=GRAY_LIGHT, lw=1.2)
style_ax(ax, 'Train − Validation Loss Gap', ylabel='Train Loss − Val Loss')
ax.text(0.98, 0.97, 'Blue = train > val   Red = val > train',
        transform=ax.transAxes, ha='right', va='top',
        fontsize=8, color=GRAY_MID)

plt.tight_layout()
plt.savefig('plot_loss_gap.png', dpi=150, bbox_inches='tight')
plt.show()

## Per-Epoch Validation Loss Improvement

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4.5))

delta_loss = [val_loss[i-1] - val_loss[i] for i in range(1, len(val_loss))]
bar_fill   = [GREEN_LIGHT if d >= 0 else RED_LIGHT for d in delta_loss]
bar_border = [GREEN       if d >= 0 else RED       for d in delta_loss]
ax.bar(epochs[1:], delta_loss, color=bar_fill, edgecolor=bar_border,
       linewidth=1.2, width=0.55)
ax.axhline(0, color=GRAY_LIGHT, lw=1.2)
style_ax(ax, 'Epoch-over-Epoch Validation Loss Improvement',
         ylabel='Δ Val Loss (positive = better)')
ax.text(0.98, 0.97, 'Green = improved   Red = worsened',
        transform=ax.transAxes, ha='right', va='top',
        fontsize=8, color=GRAY_MID)

plt.tight_layout()
plt.savefig('plot_delta_loss.png', dpi=150, bbox_inches='tight')
plt.show()

## Epoch-by-Epoch Stats Table

In [ ]:
try:
    import pandas as pd
    df = pd.DataFrame(history)
    df.columns = ['Epoch', 'Train Loss', 'Val Loss', 'Train Acc', 'Val Acc', 'Val F1']

    def highlight_best(s):
        higher_is_better = s.name in ('Train Acc', 'Val Acc', 'Val F1')
        lower_is_better  = s.name in ('Train Loss', 'Val Loss')
        if higher_is_better:
            is_best = s == s.max()
        elif lower_is_better:
            is_best = s == s.min()
        else:
            return ['' for _ in s]
        return [
            f'background-color: {GREEN_LIGHT}; color: #2D5016; font-weight: bold' if v else ''
            for v in is_best
        ]

    display(df.style
              .format({'Train Loss': '{:.4f}', 'Val Loss': '{:.4f}',
                       'Train Acc': '{:.4%}', 'Val Acc': '{:.4%}', 'Val F1': '{:.4f}'})
              .apply(highlight_best)
              .set_caption('Highlighted cells indicate the best value per column.')
              .hide(axis='index'))
except ImportError:
    header = f"{'Epoch':>6}  {'Train Loss':>10}  {'Val Loss':>10}  {'Train Acc':>10}  {'Val Acc':>10}  {'Val F1':>8}"
    print(header)
    print('-' * len(header))
    for d in history:
        print(f"{d['epoch']:>6}  {d['train_loss']:>10.4f}  {d['val_loss']:>10.4f}  "
              f"{d['train_acc']:>10.4%}  {d['val_acc']:>10.4%}  {d['val_f1']:>8.4f}")

## Summary

In [ ]:
best_epoch_loss = epochs[val_loss.index(min(val_loss))]
best_epoch_acc  = epochs[val_acc.index(max(val_acc))]
best_epoch_f1   = epochs[val_f1.index(max(val_f1))]

print('─' * 52)
print('  TRAINING SUMMARY')
print('─' * 52)
print(f'  Total epochs         : {len(epochs)}')
print(f'  Best val loss        : {min(val_loss):.4f}  (epoch {best_epoch_loss})')
print(f'  Best val accuracy    : {max(val_acc):.4%}  (epoch {best_epoch_acc})')
print(f'  Best val F1          : {max(val_f1):.4f}  (epoch {best_epoch_f1})')
print(f'  Final train/val loss : {train_loss[-1]:.4f} / {val_loss[-1]:.4f}')
print(f'  Final train/val acc  : {train_acc[-1]:.4%} / {val_acc[-1]:.4%}')
print(f'  Loss reduction       : {(val_loss[0]-min(val_loss))/val_loss[0]*100:.1f}%')
print(f'  Accuracy gain        : +{(max(val_acc)-val_acc[0])*100:.2f} pp')
print('─' * 52)